In [ ]:
import numpy as np

class Layer:
    def __init__(self, in_dim, out_dim):
        # self.W = np.random.randn(in_dim, out_dim) * np.sqrt(2 / in_dim)
        # The standard deviation is set directly via the scale argument
        self.rng =  np.random.default_rng(seed = 42)
        self.W = self.rng.normal(loc=0.0, scale=np.sqrt(2 / in_dim), size=(in_dim, out_dim))
        self.B = np.zeros((1, out_dim))
        self.X = None
        self.dW = None
        self.dB = None

    def forward(self, inputs):
        self.X = inputs
        return np.matmul(self.X, self.W) + self.B

    def backward(self, dout):
        self.dW = np.matmul(self.X.T, dout)
        self.dB = np.sum(dout, axis = 0, keepdims = True)
        return np.matmul(dout, self.W.T)

    def update(self, lr):
        self.W -= lr *  self.dW 
        self.B -=  lr *  self.dB



class Relu:
    def forward(self, Z):
        self.mask = Z > 0
        return Z * self.mask

    def backward(self, dout):
        return dout * self.mask

class Sigmoid:
    #need stable sigmoid and 

    def forward(self, Z):
        self.sigmoid_Z =  1 / (1 + np.exp(-Z))
        return self.sigmoid_Z

    def backward(self, dout):
        return dout * self.sigmoid_Z * (1 - self.sigmoid_Z)

# class Softmax:

#     def forward(self, Z):


#cross entropy
 

class MSE:
    @staticmethod
    def loss(predictions, y_true):
        return 1/2 * np.mean((y_true  - predictions) ** 2)
    
    @staticmethod
    def dpredictions(predictions, y_true):
        N = predictions.shape[0]
        return  (predictions - y_true) / N
    
class NeuralNetwork:
    def __init__(self, layers, lr = 0.01):
        self.layers = layers
        self.lr = lr

    def forward(self, inputs):
        X = inputs
        for layer in self.layers:
            X = layer.forward(X)
        return X

    def backward(self, dout):
        for layer in reversed(self.layers):
            dout = layer.backward(dout)

    def update(self):
        for layer in self.layers:
            if hasattr(layer, 'W'):
                layer.update(lr = self.lr)

    def train_step(self, X, y_true, loss_fn):
        predictions = self.forward(X)
        loss = loss_fn.loss(predictions, y_true)

        dout = loss_fn.dpredictions(predictions, y_true)
        self.backward(dout)

        self.update()
        return loss        

In [236]:
X = np.linspace(-5, 5, 1000).reshape(-1, 1)
y = 2 * (X**2) + 1

net = NeuralNetwork([
    Layer(1, 16),
    Relu(),
    Layer(16, 1)
], lr=0.01)

mse = MSE()

for epoch in range(1, 100001):
    loss = net.train_step(X, y, mse)

    if epoch % 500 == 0:
        print(epoch, loss)

500 2.6967509755838757
1000 1.6662247959644292
1500 1.423551267451092
2000 1.3839410839074855
2500 1.3782421921592103
3000 1.3773407618066955
3500 1.3772930403659076
4000 1.377290908302902
4500 1.3772908130516066
5000 1.3772908087963147
5500 1.3772908086062134
6000 1.377290808597721
6500 1.3772908085973412
7000 1.3772908085973246
7500 1.377290808597324
8000 1.3772908085973237
8500 1.3772908085973241
9000 1.3772908085973241
9500 1.3772908085973237
10000 1.377290808597324
10500 1.3772908085973246
11000 1.3772908085973237
11500 1.3772908085973232
12000 1.3772908085973232
12500 1.3772908085973232
13000 1.3772908085973232
13500 1.3772908085973232
14000 1.3772908085973232
14500 1.3772908085973232
15000 1.3772908085973232
15500 1.3772908085973232
16000 1.3772908085973232
16500 1.3772908085973232
17000 1.3772908085973232
17500 1.3772908085973232
18000 1.3772908085973232
18500 1.3772908085973232
19000 1.3772908085973232
19500 1.3772908085973232
20000 1.3772908085973232
20500 1.3772908085973232


In [240]:
net.forward(np.array([[100]])) 

array([[1313.9740852]])

In [ ]:
# # toy 1-D regression: y = 2x + 1
# X = np.linspace(-1, 1, 1000).reshape(-1, 1) * 100
# y = 2 * (X) + 1
# # print(X.dtype)
# # X[0,0] = 500
# # y[0,0] = 1001

# net = NeuralNetwork([
#     Layer(1, 16),
#     # Sigmoid(), #sigmoid is blasting when inputs are big like 500 but works for small inputs
#     Relu(),
#     Layer(16, 16),
#     Relu(),
#     Layer(16, 1)
# ], lr=0.05)

# mse = MSE()
# for epoch in range(1, 100001):
#     loss = net.train_step(X, y, mse)
#     if epoch % 50 == 0:
#         print(epoch, loss)